# Демонстрация результатов: SFT → DPO / GRPO и reward-модели

Задача — управляемая генерация рецензий Кинопоиска: модель получает префикс `Тональность: положительная/отрицательная` и начало отзыва и должна дописать рецензию нужной тональности.

| модель | что это |
|---|---|
| `base` | `Qwen/Qwen3.5-0.8B-Base` без дообучения |
| `sft` | base + LoRA после SFT на рецензиях |
| `dpo_10`, `dpo_03`, `dpo_01` | SFT + LoRA после DPO с β = 1.0 / 0.3 / 0.1 (пары собраны по RM v1) |
| `grpo_004`, `grpo_nokl` | SFT + LoRA после GRPO (300 шагов, награда — RM v1) с KL-коэффициентом β = 0.04 / 0 |
| `rm_v1`, `rm_v2` | reward-модели на `DeepPavlov/rubert-base-cased`: v1 — веса классов, v2 — веса по группам «список фильма × метка» |

### Какие файлы нужны

Проще всего — один архив **`rlhf_demo_artifacts.zip`** (адаптеры, RM, данные для оценки и сохранённые результаты).
Подойдут и исходные архивы: `sft_adapter.zip`, `reward_models.zip`, `session6.zip` (DPO), `grpo_b0.04.zip`, `grpo_b0.zip`, `kinopoisk_processed.zip`, `session5.zip`. Чего-то не хватает — соответствующая часть просто пропускается.

### Как запустить

* **Kaggle**: *Add Input* → загрузить архив как Dataset (Kaggle распакует его сам; уже существующие inputs `sft-adapter`, `rm-3`, `kinopoisk-processed` тоже найдутся). В настройках: *Accelerator — GPU T4*, *Internet — on* (нужен, чтобы скачать базовую модель с Hugging Face).
* **Colab**: положить архив в Google Drive в папку `rlhf_artifacts` (или загрузить прямо в `/content` и поставить `USE_DRIVE = False`). *Runtime → Change runtime type → T4 GPU*.
* **Локально**: положить архив рядом с ноутбуком, в папку `artifacts/` или в `~/Downloads`. Нужен Python ≥ 3.10 и установленный `torch` (https://pytorch.org); остальное ставит первая ячейка.

Без GPU всё тоже работает (в float32), только генерация заметно медленнее — размеры выборок ниже на CPU уменьшаются автоматически. На GPU нужно около 4–5 ГБ видеопамяти.

## 1. Окружение

In [ ]:
import os, sys

IN_KAGGLE = os.path.exists("/kaggle/input")
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Kaggle:", IN_KAGGLE, "| Colab:", IN_COLAB)

if IN_KAGGLE or IN_COLAB:
    %pip uninstall -y -q torchao
%pip install -q transformers==5.23.0 peft==0.22.0 accelerate==1.16.0 scikit-learn pandas pyarrow matplotlib

In [ ]:
import glob, zipfile, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers, peft
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
BASE = "Qwen/Qwen3.5-0.8B-Base"

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("устройство:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 2. Поиск файлов

Ячейка ищет известные zip-архивы, распаковывает их в папку `artifacts/` и затем находит нужные папки по характерным файлам (`adapter_config.json`, `config.json`, ...), поэтому неважно, в какой структуре лежат файлы.

In [ ]:
USE_DRIVE = True

if IN_KAGGLE:
    WORK = "/kaggle/working"
    ZIP_DIRS = [WORK]
    EXTRA_DIRS = ["/kaggle/input"]
elif IN_COLAB:
    WORK = "/content"
    ZIP_DIRS = [WORK]
    EXTRA_DIRS = []
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ZIP_DIRS.append("/content/drive/MyDrive/rlhf_artifacts")
        EXTRA_DIRS.append("/content/drive/MyDrive/rlhf_artifacts")
else:
    WORK = "."
    ZIP_DIRS = [".", "artifacts", os.path.expanduser("~/Downloads")]
    EXTRA_DIRS = []

EXTRACT_DIR = os.path.join(WORK, "artifacts")
FIND_DIRS = [EXTRACT_DIR] + EXTRA_DIRS
os.makedirs(EXTRACT_DIR, exist_ok=True)

ZIP_NAMES = ["rlhf_demo_artifacts.zip", "sft_adapter.zip", "reward_models.zip", "session5.zip",
             "session6.zip", "dpo_beta0.1.zip", "grpo_b0.04.zip", "grpo_b0.zip", "kinopoisk_processed.zip"]

zips = []
for name in ZIP_NAMES:
    for d in ZIP_DIRS:
        if os.path.exists(os.path.join(d, name)):
            zips.append(os.path.join(d, name))
            break
if len(zips) > 0 and zips[0].endswith("rlhf_demo_artifacts.zip"):
    zips = zips[:1]

for path in zips:
    target = os.path.join(EXTRACT_DIR, os.path.basename(path)[:-4])
    if os.path.exists(target):
        continue
    print("распаковываю", path)
    with zipfile.ZipFile(path) as z:
        z.extractall(target + "_tmp")
    os.rename(target + "_tmp", target)
print("архивы:", zips)

In [ ]:
def find_file(pattern):
    for root in FIND_DIRS:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if len(hits) > 0:
            return hits[0]
    return None

def find_dir(pattern):
    path = find_file(pattern)
    if path is None:
        return None
    return os.path.dirname(path)

PATHS = {
    "sft": find_dir("sft_adapter/adapter_config.json"),
    "dpo_10": find_dir("dpo_beta1.0/adapter_config.json"),
    "dpo_03": find_dir("dpo_beta0.3/adapter_config.json"),
    "dpo_01": find_dir("dpo_beta0.1/adapter_config.json"),
    "grpo_004": find_dir("grpo_b0.04/checkpoint-300/adapter_config.json"),
    "grpo_nokl": find_dir("grpo_b0/checkpoint-300/adapter_config.json"),
    "rm_v1": find_dir("rm_v1/config.json"),
    "rm_v2": find_dir("rm_v2/config.json"),
    "data": find_dir("prompts_eval.jsonl"),
}
for k, v in PATHS.items():
    print(f"{k:9s} {v if v else '--- не найдено ---'}")

assert PATHS["sft"] is not None, "не найден SFT-адаптер (sft_adapter/adapter_config.json)"
assert PATHS["rm_v1"] is not None, "не найдена reward-модель rm_v1"

## 3. Загрузка моделей

DPO и GRPO обучались как LoRA **поверх слитой SFT-модели**, поэтому SFT загружается один раз (база + SFT-адаптер, `merge_and_unload`), а все адаптеры DPO/GRPO подключаются к ней одновременно; переключение — `policy.set_adapter(...)`. Сама SFT — это та же модель с выключенными адаптерами (`policy.disable_adapter()`). Базовая модель хранится отдельной копией.

In [ ]:
LABEL = {"base": "base", "sft": "SFT",
         "dpo_10": "DPO β=1.0", "dpo_03": "DPO β=0.3", "dpo_01": "DPO β=0.1",
         "grpo_004": "GRPO β=0.04", "grpo_nokl": "GRPO β=0"}

tok = AutoTokenizer.from_pretrained(PATHS["sft"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

base = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE).to(DEVICE).eval()

sft = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE).to(DEVICE)
sft = PeftModel.from_pretrained(sft, PATHS["sft"]).merge_and_unload().eval()

ADAPTERS = []
for name in ["dpo_10", "dpo_03", "dpo_01", "grpo_004", "grpo_nokl"]:
    if PATHS[name] is not None:
        ADAPTERS.append(name)

policy = None
for name in ADAPTERS:
    if policy is None:
        policy = PeftModel.from_pretrained(sft, PATHS[name], adapter_name=name)
    else:
        policy.load_adapter(PATHS[name], adapter_name=name)
if policy is not None:
    policy.eval()

MODELS = ["base", "sft"] + ADAPTERS
print("модели:", [LABEL[m] for m in MODELS])
if DEVICE == "cuda":
    print(f"занято на GPU: {torch.cuda.memory_allocated() / 1e9:.1f} ГБ")

In [ ]:
rm_tok = AutoTokenizer.from_pretrained(PATHS["rm_v1"])
RMS = {}
for name in ["rm_v1", "rm_v2"]:
    if PATHS[name] is not None:
        RMS[name] = AutoModelForSequenceClassification.from_pretrained(PATHS[name], dtype=DTYPE).to(DEVICE).eval()
print("reward-модели:", list(RMS))

@torch.no_grad()
def rm_margin(rm, texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = rm_tok(texts[i:i + bs], truncation=True, max_length=256,
                     padding=True, return_tensors="pt").to(DEVICE)
        logits = rm(**enc).logits.float()
        out.append((logits[:, 1] - logits[:, 0]).cpu())
    return torch.cat(out).numpy()

## 4. Генерация

* `make_prompt(тональность, начало)` собирает промпт в том же формате, что при обучении.
* `generate(модель, список_промптов, ...)` — генерация одной из моделей `MODELS`. Параметры по умолчанию как при оценке в основной работе: `temperature=1.0`, без top-k/top-p, до 144 новых токенов. `temperature=0` — жадная генерация.
* reward-модель оценивает текст самой рецензии (начало из промпта + продолжение) без префикса тональности — так же, как при обучении.

In [ ]:
def make_prompt(sentiment, start=""):
    prompt = f"Тональность: {sentiment}\nОтзыв:"
    if start:
        prompt += " " + start
    return prompt

def review_text(prompt, completion):
    return prompt.split("Отзыв:", 1)[-1].strip() + completion

def distinct(texts, n):
    grams = []
    for t in texts:
        words = t.lower().split()
        grams += [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]
    return len(set(grams)) / max(1, len(grams))

def rep4(ids):
    ids = list(ids)
    grams = [tuple(ids[i:i + 4]) for i in range(len(ids) - 3)]
    return 1 - len(set(grams)) / max(1, len(grams))

@torch.no_grad()
def generate(model_name, prompts, max_new_tokens=144, temperature=1.0, seed=0):
    torch.manual_seed(seed)
    enc = tok(prompts, return_tensors="pt", padding=True).to(DEVICE)
    params = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    if temperature > 0:
        params.update(do_sample=True, temperature=temperature, top_p=1.0, top_k=0)
    else:
        params.update(do_sample=False)

    if model_name == "base":
        out = base.generate(**enc, **params)
    elif model_name == "sft" and policy is None:
        out = sft.generate(**enc, **params)
    elif model_name == "sft":
        with policy.disable_adapter():
            out = policy.generate(**enc, **params)
    else:
        policy.set_adapter(model_name)
        out = policy.generate(**enc, **params)

    rows = []
    for ids in out[:, enc["input_ids"].shape[1]:].tolist():
        ended = tok.eos_token_id in ids
        if ended:
            ids = ids[:ids.index(tok.eos_token_id) + 1]
        rows.append({"completion": tok.decode(ids, skip_special_tokens=True),
                     "len_tok": len(ids), "ended": ended, "ids": ids})
    return rows

def compare_models(sentiment, start="", n=2, models=None, max_new_tokens=144, temperature=1.0, seed=0):
    if models is None:
        models = MODELS
    prompt = make_prompt(sentiment, start)
    sign = 1 if sentiment == "положительная" else -1
    print("ПРОМПТ:", repr(prompt))
    for name in models:
        gens = generate(name, [prompt] * n, max_new_tokens, temperature, seed)
        reviews = [review_text(prompt, g["completion"]) for g in gens]
        margins = rm_margin(RMS["rm_v1"], reviews)
        print(f"\n===== {LABEL[name]} =====")
        for g, text, m in zip(gens, reviews, margins):
            verdict = "в цель" if sign * m > 0 else "мимо"
            eos = ", EOS" if g["ended"] else ", обрезано"
            print(f"[rm_v1 {m:+.2f} ({verdict}) | {g['len_tok']} ток{eos}]")
            print(text, "\n")

### 4.1 Одна затравка — все модели

Меняйте тональность (`"положительная"` / `"отрицательная"`) и начало отзыва. Интереснее всего «против затравки»: начало явно хвалит фильм, а просим отрицательный отзыв (или наоборот).

In [ ]:
compare_models("отрицательная", "Великолепная работа!!! Я", n=2)

In [ ]:
compare_models("положительная", "Скучный и затянутый", n=2)

Можно генерировать и отдельной моделью, в том числе по произвольному промпту:

In [ ]:
for g in generate("sft", [make_prompt("положительная", "Пошёл на этот фильм")] * 3, temperature=0.8, seed=1):
    print(g["completion"], "\n")

### 4.2 Сравнение моделей на eval-промптах

Берём промпты из тестовой части (`prompts_eval.jsonl`, поровну на каждую тональность), генерируем одним и тем же сидом каждой моделью и считаем:

* `rm1_hit` / `rm2_hit` — доля ответов, где RM согласна с целевой тональностью;
* `rm1_mean` — средний margin RM v1 со знаком цели (больше — увереннее «в цель»);
* `rm1_ctrl_auc` — AUC разделения целевых тональностей по margin RM v1;
* `distinct2` — доля уникальных биграмм слов (разнообразие), `len_tok` — длина ответа, `ended` — доля ответов, закончившихся EOS.

Это быстрая проверка на небольшой выборке; полная оценка (1000 ответов, судья Qwen3.5-4B, KL, перплексия) — в разделе 6.

In [ ]:
N_PROMPTS = 64 if DEVICE == "cuda" else 16

if PATHS["data"] is not None:
    pe = pd.read_json(os.path.join(PATHS["data"], "prompts_eval.jsonl"), lines=True)
    pe = pe.groupby("target").head(N_PROMPTS // 2).reset_index(drop=True)
else:
    starts = ["Фильм", "Честно говоря,", "Пошёл в кино на", "Этот фильм я", "Давно хотел посмотреть"]
    pe = pd.DataFrame([{"prompt": make_prompt(s, st), "target": t}
                       for t, s in ((1, "положительная"), (0, "отрицательная")) for st in starts])
print(len(pe), "промптов,", pe["target"].value_counts().to_dict())

summary, all_gens = [], {}
for name in MODELS:
    gens = pd.DataFrame(generate(name, pe["prompt"].tolist(), seed=0))
    gens["target"] = pe["target"].values
    gens["review"] = [review_text(p, c) for p, c in zip(pe["prompt"], gens["completion"])]
    sign = np.where(gens["target"] == 1, 1.0, -1.0)
    row = {"model": LABEL[name]}
    raw = rm_margin(RMS["rm_v1"], gens["review"].tolist())
    gens["rm_v1"] = sign * raw
    row["rm1_hit"] = (gens["rm_v1"] > 0).mean()
    row["rm1_mean"] = gens["rm_v1"].mean()
    row["rm1_ctrl_auc"] = roc_auc_score(gens["target"], raw)
    if "rm_v2" in RMS:
        gens["rm_v2"] = sign * rm_margin(RMS["rm_v2"], gens["review"].tolist())
        row["rm2_hit"] = (gens["rm_v2"] > 0).mean()
    row["distinct2"] = distinct(gens["completion"], 2)
    row["len_tok"] = gens["len_tok"].mean()
    row["ended"] = gens["ended"].mean()
    summary.append(row)
    all_gens[name] = gens
    print("готово:", LABEL[name])

summary = pd.DataFrame(summary).set_index("model").round(3)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
summary["rm1_hit"].plot.bar(ax=axes[0], color="tab:blue")
axes[0].set_title("Доля ответов нужной тональности (по RM v1)")
axes[0].set_ylim(0, 1.05)
summary["distinct2"].plot.bar(ax=axes[1], color="tab:orange")
axes[1].set_title("Разнообразие: distinct-2")
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Ответы разных моделей на один и тот же промпт из этой выборки:

In [ ]:
i = 0
print("ПРОМПТ:", repr(pe.loc[i, "prompt"]), "| цель:", "положительная" if pe.loc[i, "target"] == 1 else "отрицательная")
for name in MODELS:
    r = all_gens[name].loc[i]
    print(f"\n===== {LABEL[name]} | rm_v1 {r['rm_v1']:+.2f} | {r['len_tok']} ток =====")
    print(r["review"])

## 5. Reward-модели

`margin = logit(good) − logit(bad)`, `p_good = sigmoid(margin)`. В GRPO наградой был `margin` RM v1 со знаком целевой тональности.

### 5.1 Свои тексты

In [ ]:
pd.set_option("display.max_colwidth", 120)

def rm_table(texts):
    table = pd.DataFrame({"text": texts})
    for name, rm in RMS.items():
        m = rm_margin(rm, texts)
        table[name] = m.round(2)
        table[name + " p_good"] = (1 / (1 + np.exp(-m))).round(3)
    return table

my_texts = [
    "Отличный фильм, прекрасная игра актёров и трогательный сюжет. Обязательно пересмотрю.",
    "Скучный фильм, слабая игра актёров и предсказуемый сюжет. Зря потратил время.",
    "Фильм неплохой, но второй раз смотреть не буду.",
    "Не понимаю восторгов: красивая картинка, но за ней пустота.",
]
rm_table(my_texts)

### 5.2 Стресс-тесты (как в основной работе)

In [ ]:
stress = {
    "обычный хороший": "Отличный фильм, прекрасная игра актёров и трогательный сюжет. Обязательно пересмотрю.",
    "обычный плохой": "Скучный фильм, слабая игра актёров и предсказуемый сюжет. Зря потратил время.",
    "повтор 'отлично'": "отлично " * 20,
    "повтор 'ужасно'": "ужасно " * 20,
    "обрывок": "Фильм",
    "случайные слова": "стол облако бежать синий вчера карандаш река громко",
    "английский": "This movie was absolutely wonderful, great acting and a moving story.",
    "противоречие": "Отличный фильм, прекрасная игра актёров. Шучу, ничего хуже я не видел.",
}
if PATHS["data"] is not None:
    test = pd.read_json(os.path.join(PATHS["data"], "rm_test.jsonl"), lines=True)
    top = test.loc[test["part"] == "top250", "movie_name"].drop_duplicates().sample(3, random_state=0)
    bottom = test.loc[test["part"] == "bottom100", "movie_name"].drop_duplicates().sample(3, random_state=0)
    stress["названия TOP250"] = ". ".join(top)
    stress["названия BOTTOM100"] = ". ".join(bottom)

table = rm_table(list(stress.values()))
table.insert(0, "тест", list(stress.keys()))
table.drop(columns="text")

### 5.3 Качество на тестовой выборке

Тест — рецензии из фильмов, не попавших в train (сплит по фильмам). Отдельно смотрим AUC внутри `top250` и `bottom100`: там метка расходится с «репутацией» фильма, и видно, насколько RM оценивает текст, а не узнаёт фильм.

In [ ]:
N_TEST = None if DEVICE == "cuda" else 600

if PATHS["data"] is None:
    print("нет rm_test.jsonl — пропускаю")
else:
    test = pd.read_json(os.path.join(PATHS["data"], "rm_test.jsonl"), lines=True)
    if N_TEST is not None:
        test = test.sample(N_TEST, random_state=0).reset_index(drop=True)
    print(len(test), "рецензий |", test["part"].value_counts().to_dict(), "| доля хороших:", round(test["label"].mean(), 3))

    rows = []
    for name, rm in RMS.items():
        test[name] = rm_margin(rm, test["text"].tolist())
        pred = (test[name] > 0).astype(int)
        row = {"model": name,
               "AUC": roc_auc_score(test["label"], test[name]),
               "accuracy": accuracy_score(test["label"], pred),
               "f1_macro": f1_score(test["label"], pred, average="macro")}
        for part in ("top250", "bottom100"):
            g = test[test["part"] == part]
            row[f"AUC {part}"] = roc_auc_score(g["label"], g[name])
        for lbl, lbl_name in ((0, "bad"), (1, "good")):
            g = test[test["label"] == lbl]
            row[f"acc {lbl_name}"] = ((g[name] > 0).astype(int) == lbl).mean()
        rows.append(row)
    display(pd.DataFrame(rows).set_index("model").round(3))

In [ ]:
if PATHS["data"] is not None:
    for name in RMS:
        print(name)
        print(pd.DataFrame(confusion_matrix(test["label"], (test[name] > 0).astype(int)),
                           index=["true bad", "true good"], columns=["pred bad", "pred good"]), "\n")

    fig, axes = plt.subplots(1, len(RMS), figsize=(6 * len(RMS), 3.5), squeeze=False)
    for ax, name in zip(axes[0], RMS):
        for lbl, lbl_name in ((1, "good"), (0, "bad")):
            ax.hist(test.loc[test["label"] == lbl, name], bins=40, alpha=0.6, density=True, label=lbl_name)
        ax.axvline(0, color="black", lw=1)
        ax.set_title(f"{name}: распределение margin на тесте")
        ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Сохранённые результаты полной оценки

Эта часть не требует моделей — только сохранённые при обучении файлы `eval_*.parquet` (по 1000 ответов на 500 eval-промптов) и логи GRPO. Колонки:

* `kl` — KL(политика ‖ SFT) на ответ, в натах (считался по log-вероятностям ответов);
* `judge_*` — оценка независимого судьи **Qwen3.5-4B** (margin log p(«положительная») − log p(«отрицательная») со знаком цели), `judge_ctrl_auc` — насколько хорошо по судье различаются две целевые тональности;
* `rm1_mean` — средний margin RM v1 со знаком цели;
* `distinct2` — разнообразие, `rep4` — доля повторяющихся 4-грамм токенов внутри ответа, `len_tok`, `ended` — длина и доля ответов с EOS.

In [ ]:
saved_files = {"SFT": "eval_sft.parquet",
               "DPO β=1.0": "eval_dpo_beta1.0.parquet",
               "DPO β=0.3": "eval_dpo_beta0.3.parquet",
               "DPO β=0.1": "eval_dpo_beta0.1.parquet"}
for step in range(50, 301, 50):
    saved_files[f"GRPO β=0.04 s{step}"] = f"eval_grpo_b0.04_s{step}.parquet"
for step in range(50, 301, 50):
    saved_files[f"GRPO β=0 s{step}"] = f"eval_grpo_b0_s{step}.parquet"

saved, rows = {}, []
for name, fname in saved_files.items():
    path = find_file(fname)
    if path is None:
        continue
    g = pd.read_parquet(path)
    saved[name] = g
    sign = np.where(g["target"] == 1, 1, -1)
    rows.append({
        "model": name,
        "kl": g["kl"].mean(),
        "judge_hit": (g["j"] > 0).mean(),
        "judge_mean": g["j"].mean(),
        "judge_ctrl_auc": roc_auc_score(g["target"], g["j"] * sign),
        "rm1_mean": g["r1"].mean(),
        "rm1_hit": (g["r1"] > 0).mean(),
        "distinct2": distinct(g["completion"], 2),
        "rep4": np.mean([rep4(x) for x in g["comp_ids"]]),
        "len_tok": np.mean([len(x) for x in g["comp_ids"]]),
        "ended": g["ended"].mean(),
    })

if len(rows) == 0:
    print("сохранённых eval_*.parquet не найдено — раздел пропускается")
else:
    final = pd.DataFrame(rows).set_index("model")
    final["kl_per_tok"] = final["kl"] / final["len_tok"]
    display(final.round(3))

In [ ]:
if len(saved) > 0:
    groups = {
        "DPO (β = 1.0 → 0.1)": final[final.index.str.startswith("DPO")].sort_values("kl"),
        "GRPO β = 0.04 (шаги 50 → 300)": final[final.index.str.startswith("GRPO β=0.04")],
        "GRPO β = 0 (шаги 50 → 300)": final[final.index.str.startswith("GRPO β=0 ")],
    }
    panels = [("judge_mean", "Средний margin судьи (со знаком цели)"),
              ("distinct2", "Разнообразие: distinct-2"),
              ("rep4", "Повторы внутри ответа (rep4)")]

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    for ax, (col, title) in zip(axes, panels):
        if "SFT" in final.index:
            ax.scatter([final.loc["SFT", "kl"] + 1], [final.loc["SFT", col]], marker="*", s=200,
                       c="black", label="SFT", zorder=3)
        for lbl, d in groups.items():
            if len(d) > 0:
                ax.plot(d["kl"] + 1, d[col], marker="o", label=lbl)
        ax.set_xscale("log")
        ax.set_xlabel("KL до SFT + 1, наты на ответ (лог. шкала)")
        ax.set_title(title)
        ax.grid(alpha=0.3)
    axes[0].legend()
    plt.tight_layout()
    plt.show()

### 6.1 Ход обучения GRPO

In [ ]:
logs = {}
for beta, fname in (("0.04", "grpo_b0.04_log.csv"), ("0", "grpo_b0_log.csv")):
    path = find_file(fname)
    if path is not None:
        log = pd.read_csv(path)
        logs[beta] = log[log["reward"].notna()]

if len(logs) == 0:
    print("логов GRPO не найдено")
else:
    panels = [("reward", "Награда (margin RM v1 со знаком цели)"),
              ("kl", "KL к SFT на токен (при β = 0 тренер его не логирует)"),
              ("completions/mean_length", "Средняя длина ответа, токенов")]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, (col, title) in zip(axes, panels):
        for beta, log in logs.items():
            if col in log.columns:
                ax.plot(log["step"], log[col].rolling(10, min_periods=1).mean(), label=f"β = {beta}")
        ax.set_xlabel("шаг")
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

### 6.2 Ответы из сохранённых генераций

Во всех `eval_*.parquet` промпты идут в одном порядке, поэтому строку `i` можно сравнить между моделями. Колонки `j` и `r1` — margin судьи и RM v1 со знаком цели.

In [ ]:
SHOW = ["SFT", "DPO β=1.0", "DPO β=0.1", "GRPO β=0.04 s300", "GRPO β=0 s300"]

i = 2
if len(saved) > 0:
    first = saved[next(iter(saved))]
    print("ПРОМПТ:", repr(first.loc[i, "prompt"]), "| цель:", "положительная" if first.loc[i, "target"] == 1 else "отрицательная")
    for name in SHOW:
        if name in saved:
            r = saved[name].loc[i]
            print(f"\n===== {name} | судья {r['j']:+.2f} | rm_v1 {r['r1']:+.2f} | {len(r['comp_ids'])} ток =====")
            print(r["review"])

In [ ]:
from collections import Counter

def top_words(df, target, k=15):
    text = " ".join(df.loc[df["target"] == target, "completion"]).lower()
    words = [w for w in re.findall(r"[а-яё]+", text) if len(w) > 3]
    return [w for w, _ in Counter(words).most_common(k)]

for t, t_name in ((1, "положительная"), (0, "отрицательная")):
    print(f"\n=== самые частые слова, цель: {t_name} ===")
    for name in SHOW:
        if name in saved:
            print(f"{name:17s}", ", ".join(top_words(saved[name], t)))

### 6.3 Примеры пар для DPO

In [ ]:
path = find_file("dpo_pairs.parquet")
if path is None:
    print("dpo_pairs.parquet не найден")
else:
    pairs = pd.read_parquet(path)
    print(len(pairs), "пар | средний разрыв наград:", round(pairs["margin"].mean(), 2))
    for _, r in pairs.sample(2, random_state=0).iterrows():
        print(f"\n[цель={r['target']}]  chosen {r['r_chosen']:+.2f} | rejected {r['r_rejected']:+.2f}")
        print("ПРОМПТ: ", r["prompt"].split("Отзыв:", 1)[1])
        print("CHOSEN: ", r["chosen"][:300])
        print("REJECT: ", r["rejected"][:300])